# Sprint 2 — MVP analítico de previsão

Este notebook transforma a base municipal do Atlas da Violência em um painel temporal e testa um primeiro problema supervisionado: prever a variação da taxa de homicídios do município no ano seguinte. A coorte principal considera municípios com pelo menos 50 mil habitantes no ano de origem; a base completa permanece disponível para comparação.

O alvo principal é `100 × (log1p(taxa futura) - log1p(taxa atual))`. Essa transformação permite representar crescimento e queda em uma escala mais estável, inclusive quando a taxa atual é zero. O resultado também é convertido para uma porcentagem estabilizada para facilitar a interpretação.

## Decisões metodológicas

- Unidade de previsão: município-ano.
- Horizonte: um ano à frente.
- Variáveis iniciais: ano, taxa atual, duas defasagens, média e desvio-padrão dos últimos três anos, variação logarítmica atual e log da população.
- Divisão temporal: treino até 2016, validação de 2017 a 2019 e teste a partir de 2020.
- Modelos: baseline de variação zero, baseline do último valor, Ridge e Random Forest.

A divisão temporal evita que anos futuros influenciem o treinamento e será mantida nas próximas comparações. O filtro populacional é aplicado antes da divisão, usando somente a população do ano de origem.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.models import evaluate_by_population_size, run_experiment
from src.population import filter_by_population, load_population
from src.prepare_data import (
    FEATURE_COLUMNS,
    TARGET_LOG_COLUMN,
    TARGET_PERCENT_COLUMN,
    build_model_dataset,
    load_homicide_rates,
    split_by_year,
)

RAW_PATH = ROOT / 'data' / 'raw' / 'taxa_homicidios_2024.csv'
POPULATION_PATH = ROOT / 'data' / 'raw' / 'populacao_municipal_ibge_2001_2021.csv'

In [ ]:
rates = load_homicide_rates(RAW_PATH)
population = load_population(POPULATION_PATH)

print(f'Linhas: {len(rates):,}')
print(f'Municípios: {rates["municipio_codigo"].nunique():,}')
print(f'Período: {rates["ano"].min()}–{rates["ano"].max()}')
rates.head()

## Construção do painel supervisionado

A preparação só mantém uma linha quando os três anos históricos e o ano futuro são consecutivos. Assim, a ausência estrutural de 1996–1999 não é tratada como se fosse uma passagem anual normal.

In [ ]:
model_data_full = build_model_dataset(rates, population=population)
model_data = filter_by_population(model_data_full, minimum=50_000)

print(f'Linhas na base completa: {len(model_data_full):,}')
print(f'Linhas na coorte principal (população ≥ 50 mil): {len(model_data):,}')
print(f'Municípios com janela válida: {model_data["municipio_codigo"].nunique():,}')
print(f'Anos de origem: {model_data["ano"].min()}–{model_data["ano"].max()}')
print(f'Taxa de linhas com variação percentual bruta válida: {model_data[TARGET_PERCENT_COLUMN].notna().mean():.1%}')
print(f'População observada na origem: {model_data["populacao_observada"].mean():.1%}')
model_data[FEATURE_COLUMNS + [TARGET_LOG_COLUMN, TARGET_PERCENT_COLUMN]].describe().T

In [ ]:
splits = split_by_year(model_data)

pd.DataFrame(
    [
        {
            'divisao': name,
            'linhas': len(split),
            'ano_inicial': split['ano'].min(),
            'ano_final': split['ano'].max(),
            'municipios': split['municipio_codigo'].nunique(),
        }
        for name, split in splits.items()
    ]
)

## Baselines e modelos iniciais

O primeiro baseline prevê sempre nenhuma variação; o segundo repete a última variação observada. Um modelo só é útil se superar esses pontos de referência no conjunto de teste. Ridge oferece uma relação linear interpretável; Random Forest representa relações não lineares, mantendo a complexidade controlada pelo tamanho mínimo das folhas. A base completa é executada novamente como análise de sensibilidade.

In [ ]:
metrics, predictions = run_experiment(model_data)
full_metrics, _ = run_experiment(model_data_full)

test_metrics = (
    metrics[metrics['divisao'].eq('test')]
    .sort_values('mae_log_100')
)
test_metrics

In [ ]:
comparison_columns = [
    'modelo', 'mae_log_100', 'rmse_log_100', 'medae_log_100',
    'r2_log_100', 'mae_percentual', 'medae_percentual'
]
full_test_metrics = full_metrics[full_metrics['divisao'].eq('test')].sort_values('mae_log_100')
size_metrics = evaluate_by_population_size(predictions)
display(test_metrics[comparison_columns].round(3))
display(full_test_metrics[comparison_columns].round(3))
size_metrics.sort_values(['modelo', 'porte_populacional']).round(3)

In [ ]:
# Inspeção visual de um município no período de teste.
sample_city = predictions.loc[
    predictions['divisao'].eq('test'), 'municipio_codigo'
].iloc[0]
sample = predictions[
    predictions['municipio_codigo'].eq(sample_city)
    & predictions['divisao'].eq('test')
]

plot_data = sample.pivot(
    index='ano', columns='modelo', values='previsao_percentual_estabilizada'
)
ax = plot_data.plot(marker='o', figsize=(10, 4), title=f'Previsões — município {sample_city}')
sample.set_index('ano')['alvo_percentual'].plot(ax=ax, color='black', linestyle='--', label='observado')
ax.set_ylabel('Variação percentual')
ax.legend()

## Leitura inicial

A comparação no teste deve orientar a próxima sprint, mas não constitui validação definitiva do modelo. O conjunto ainda usa apenas o histórico da própria taxa; faltam variáveis socioeconômicas, demográficas e de segurança pública. Também será necessário comparar diferentes horizontes, avaliar estabilidade por região e tratar explicitamente municípios com taxa igual a zero antes do artigo científico.

O próximo passo é selecionar o modelo e ampliar as variáveis sem alterar a divisão temporal.